In [1]:
import os
print("Current working directory:", os.getcwd())

Current working directory: /Users/nullset/Developer/ML/learning-project/notebooks


In [2]:
import sys, os
# Manually set your repo root (DO THIS ONCE)
repo_root = "/Users/nullset/Developer/ML/learning-project"

print("Repo root:", repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src import features

Repo root: /Users/nullset/Developer/ML/learning-project


In [3]:
raw_path = os.path.join(repo_root, "data", "raw")
interim_path = os.path.join(repo_root, "data", "interim")

features.process_folder(raw_path, interim_path, augment=True)

In [6]:
import os
import shutil
import random

processed_path = os.path.join(repo_root, "data", "processed")

def split_dataset(interim_dir, processed_dir, train_ratio=0.7, val_ratio=0.15):
    os.makedirs(processed_dir, exist_ok=True)

    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(processed_dir, split), exist_ok=True)

    for class_name in os.listdir(interim_dir):
        class_path = os.path.join(interim_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        files = os.listdir(class_path)
        random.shuffle(files)

        n = len(files)
        n_train = int(train_ratio * n)
        n_val = int(val_ratio * n)

        splits = {
            "train": files[:n_train],
            "val": files[n_train:n_train+n_val],
            "test": files[n_train+n_val:]
        }

        for split, split_files in splits.items():
            split_class_dir = os.path.join(processed_dir, split, class_name)
            os.makedirs(split_class_dir, exist_ok=True)

            for f in split_files:
                src = os.path.join(class_path, f)
                dst = os.path.join(split_class_dir, f)
                shutil.copy(src, dst)

# Run it
split_dataset(interim_path, processed_path)
print("Dataset split complete")

Dataset split complete


In [7]:
import numpy as np

def load_npy_dataset(folder):
    X = []
    y = []

    for class_name in os.listdir(folder):
        class_path = os.path.join(folder, class_name)
        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            if file.endswith(".npy"):
                X.append(np.load(os.path.join(class_path, file)))
                y.append(int(class_name))

    return np.array(X), np.array(y)

X_train, y_train = load_npy_dataset(os.path.join(processed_path, "train"))
X_val, y_val = load_npy_dataset(os.path.join(processed_path, "val"))

In [8]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(64,64,3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')  # 3 classes
])

In [9]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step - accuracy: 0.3810 - loss: 1.2385 - val_accuracy: 0.3333 - val_loss: 1.0981
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5476 - loss: 1.0639 - val_accuracy: 0.3333 - val_loss: 1.1447
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3333 - loss: 1.1236 - val_accuracy: 0.4444 - val_loss: 1.0875
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5000 - loss: 1.0426 - val_accuracy: 0.3333 - val_loss: 1.1058
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.3333 - loss: 1.0409 - val_accuracy: 0.3333 - val_loss: 1.0269
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4048 - loss: 0.9409 - val_accuracy: 0.5556 - val_loss: 0.9841
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9286 - loss: 0.8746 - val_accuracy: 0.5556 - val_loss: 0.9232
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7857 - loss: 0.7938 - val_accuracy: 0.6667 - val_loss: 0.8480

In [10]:
# Model shifted to models/

model.save(os.path.join(repo_root, "models", "hand_sign_cnn.h5"))

In [11]:
X_test, y_test = load_npy_dataset(os.path.join(processed_path, "test"))

model.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6000 - loss: 0.8547


[0.8547004461288452, 0.6000000238418579]

In [21]:
sample = X_test[1]
pred = model.predict(sample.reshape(1,64,64,3))

print("Predicted:", np.argmax(pred))
print("Actual:", y_test[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Predicted: 1
Actual: 0
